# Eval de l'OCR-VLM maison sur des tickets reels (Kaggle T4)

Lance `scripts/evaluate_ocr_vlm.py` sur le GPU. Surtout pas en local : l'eval fait une
generation autoregressive sur des centaines de tickets, et ca a deja fige la machine.

Tous les `ocr_vlm_epoch*.pt` trouves sont evalues, ce qui permet de comparer deux epochs
directement (40 contre 50, par exemple).

A attacher en Input : le bundle de code, `receipt-vlm-real-data` (les `raw/` et `labels/`)
et `receipt-vlm-eval-ckpts` (les checkpoints et le tokenizer).

GPU T4, internet active.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# On materialise le bundle de code depuis /kaggle/input
import glob, os, shutil, zipfile
from pathlib import Path
WORK = Path("/kaggle/working/repo")
def materialize():
    zips = glob.glob("/kaggle/input/**/receipt_vlm_colab_bundle.zip", recursive=True)
    if zips:
        WORK.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(WORK)
        return
    hits = glob.glob("/kaggle/input/**/vlm_training/scripts/evaluate_ocr_vlm.py", recursive=True)
    if hits:
        dest = WORK / "dev_ocr"
        if not dest.exists():
            shutil.copytree(Path(hits[0]).resolve().parents[2], dest)
        return
    raise FileNotFoundError("code bundle not found in /kaggle/input -- Add Input the rebuilt bundle")
if not list(WORK.glob("**/vlm_training/scripts/evaluate_ocr_vlm.py")):
    materialize()
hits = glob.glob(str(WORK / "**/vlm_training/scripts/evaluate_ocr_vlm.py"), recursive=True)
assert hits, ("evaluate_ocr_vlm.py not found -- your uploaded bundle predates the read-acc eval; "
              "rebuild with scripts/zip_selfcontained_colab.py and re-upload")
TRAIN_PKG = Path(hits[0]).resolve().parents[1]
DEV_OCR = TRAIN_PKG.parent
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)

In [ ]:
# Dependances
import subprocess, sys
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("-r", "requirements-training.txt", "tokenizers>=0.22,<=0.23")
pip("-e", str(DEV_OCR))     # receipt_ocr
pip("-e", str(TRAIN_PKG))   # receipt_vlm
print("Install OK")

In [ ]:
# On cherche les donnees et les checkpoints. Kaggle imbrique ses entrees d'un ou deux
# niveaux, donc recherche recursive.
import glob, os
def _find_data_base():
    for labels in glob.glob("/kaggle/input/**/labels", recursive=True):
        if os.path.isdir(os.path.join(os.path.dirname(labels), "raw")):
            return os.path.dirname(labels)
    return None
REAL_DATA_DIR = _find_data_base()
CKPTS = sorted(glob.glob("/kaggle/input/**/ocr_vlm_epoch*.pt", recursive=True))
TOK   = (glob.glob("/kaggle/input/**/tokenizer.json", recursive=True) or [None])[0]
print("REAL_DATA_DIR ->", REAL_DATA_DIR)
print("TOKENIZER     ->", TOK)
print("CHECKPOINTS   ->", [os.path.basename(c) for c in CKPTS])
assert REAL_DATA_DIR and CKPTS and TOK, "attach the code bundle + real-data + eval-ckpts datasets"

In [ ]:
# On evalue chaque checkpoint trouve, ce qui donne le avant/apres directement
import subprocess, sys, os
DATASETS  = ["wildreceipt", "trainingdatapro"]
SYNTHETIC = 128
for ckpt in CKPTS:
    print()
    print("=" * 72)
    print("EVAL", os.path.basename(ckpt))
    print("=" * 72, flush=True)
    subprocess.run([sys.executable, "-u", "scripts/evaluate_ocr_vlm.py",
                    "--checkpoint", ckpt, "--tokenizer", TOK, "--data-dir", REAL_DATA_DIR,
                    "--datasets", *DATASETS, "--synthetic", str(SYNTHETIC),
                    "--examples", "3", "--batch-size", "32"], check=True)
